# Processing model payoffs

**Models:**
- Main comparisons: `ours_full` (Intuitive Gamer [main model; i.e., with full simulations to termination]), `expert` (Expert Gamer), `expert_mcts` (MCTS), `depth3` (Depth-3 version of IGM),
  `random_terminal` (Random Gamer), `offense` (Ablation; goal progress only), `defense` (Ablation; goal blocking only), `ours` (partial IGM -- i.e., not necessarily to end), 
- LLMs (just in Supp): `gpt_nocot`, `gpt_cot`, `llama3170b-vanilla`, `llama3170b-cot`, `gpt-o1`

**Outputs:**
- `final_processed_res/think/model_payoff.json` — dict keyed by model, then by
  game; symbolic entries have `payoff/pwin/pdraw/sims`, LLM entries have `payoff`.
- `final_processed_res/think/payoff_cat_dists.csv` — per-game |model - human|
  payoff distances, by game category, for downstream box plots.
- `final_processed_res/think/llm_funness.json` — `{model: {game: [scores]}}` of
  raw per-rollout LLM fun ratings (used downstream for LLM-vs-human fun comparisons).

In [ ]:
import os
import json
from ast import literal_eval
from math import sqrt

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error

import analysis_utils
from analysis_utils import (
    by_participants, map_score_to_judgment,
    compute_utility, get_pwin, process_model_runs,
)
from constants import THINK_HUMAN_DATA, THINK_STIMULI_PTH

MODEL_DIR = "../model-data/think-exp"
INTERMODEL_DIR = "../model-data/intermodel"
LLM_DIR = f"{MODEL_DIR}/llms"
OUT_DIR = "final_processed_res/think"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
_, idx2game, game2idx, _ = analysis_utils.process_game_stimuli(THINK_STIMULI_PTH)
human_df = pd.read_csv(THINK_HUMAN_DATA)
all_game_types, _ = analysis_utils.get_game_categories(human_df)
ordered_games = sorted(set(human_df.game_id))

## Load symbolic + GPT model rollouts

In [ ]:
model2pth = {
    "ours":             f"{MODEL_DIR}/full_or_partial/partial_with_draw.txt",
    "ours_full":        f"{MODEL_DIR}/full_or_partial/full.txt",
    "depth3":           f"{MODEL_DIR}/heuristics_depth_results_merged.txt",
    "random_terminal":  f"{MODEL_DIR}/main_rand_untilend_results_400.txt",
    "expert":           f"{MODEL_DIR}/2025-07-04_heuristic_search_eg.txt",
    "expert_mcts":      f"{MODEL_DIR}/mcts_results_merged.txt",
    "offense":          f"{MODEL_DIR}/ablations/heuristics/{{'center_weight': 1.0, 'block_weight': 0.0, 'connect_weight': 1.0}}.txt",
    "defense":          f"{MODEL_DIR}/ablations/heuristics/{{'center_weight': 1.0, 'block_weight': 1.0, 'connect_weight': 0.0}}.txt",
    "gpt_cot":          f"{LLM_DIR}/gpt_responses_advantage/agg_advantage_True_20240130-2159.json",
    "gpt_nocot":        f"{LLM_DIR}/gpt_responses/agg_advantage_False_20240131-0023.json",
    "gpt_cot_fun":      f"{LLM_DIR}/gpt_responses_fun/agg_how_fun_True_20240130-2253.json",
    "gpt_nocot_fun":    f"{LLM_DIR}/gpt_responses/agg_how_fun_False_20240131-0029.json",
}

model2res = {}
for model, pth in model2pth.items():
    res = process_model_runs(model, pth, idx2game)
    if res is None:
        raise FileNotFoundError(f"Failed to load {model} from {pth}")
    model2res[model] = res


Load external LLMs 

In [ ]:
external_llms = {
    "llama3170b-vanilla": "vanilla-llama-orig121-eval-score_llm_fun-score_llm_fair",
    "llama3170b-cot":     "llama3170bcot-orig121-eval-score_llm_fun-score_llm_fair",
    "gpt-o1":             "run-o1-orig121-eval-score_llm_fun-score_llm_fair",
}

def load_external_llm_scores(folder):
    # Aggregates per-game `score_llm_fair` (payoff) and `score_llm_fun` (funness)
    # values across all run*_score_data.json files. Returns (payoffs, funness)
    # with payoffs in the schema:
    # payoffs:  {game_id -> {'payoff': [scores]}}
    # funness:  {game_id -> [scores]}  (fun==-1 dropped).
    fdir = f"{LLM_DIR}/{folder}"
    payoffs, funness = {}, {}
    for fname in os.listdir(fdir):
        if "score_data" not in fname:
            continue
        with open(f"{fdir}/{fname}") as f:
            rollout = json.load(f)
        for k, entry in rollout.items():
            if not k.startswith("game_"):
                continue
            tag = entry["game_tag"]
            n_rows, n_cols, win_conds = tag.split("*")
            if "Inf" in n_rows:
                tag = tag.replace("Inf", "inf")
            else:
                tag = f"{float(int(n_rows))}*{float(int(n_cols))}*{win_conds}"
            fair = entry["game_scores"]["score_llm_fair"]
            fun = entry["game_scores"]["score_llm_fun"]
            payoffs.setdefault(tag, {"payoff": []})["payoff"].append(fair)
            if fun != -1:
                funness.setdefault(tag, []).append(fun)
    return payoffs, funness

external_llm_payoffs = {}
external_llm_funness = {}
for model_tag, folder in external_llms.items():
    fdir = f"{LLM_DIR}/{folder}"
    if not os.path.isdir(fdir):
        print(f"Skipping {model_tag} — missing {fdir}")
        continue
    external_llm_payoffs[model_tag], external_llm_funness[model_tag] = (
        load_external_llm_scores(folder)
    )
    print(f"Loaded {model_tag}: "
          f"{len(external_llm_payoffs[model_tag])} games payoff, "
          f"{len(external_llm_funness[model_tag])} games funness")

Build per-game payoff dict for each model


In [ ]:
EXPERT_LIKE = {"expert", "expert_mcts", "depth3"}

def per_participant_judgments(scores, model):
    m, n = (5, 10) if model in EXPERT_LIKE else (20, 20)
    try:
        groups = by_participants(scores, m=m, n=n)
    except AssertionError:
        # fall back to one big group if the run produced a non-(m*n)-length list
        groups = by_participants(scores, m=1, n=len(scores))
    judgments = map_score_to_judgment(groups)
    payoff = [j["exp_util"] for j in judgments]
    pwin = [grp.count(1) / len(grp) for grp in groups]
    pdraw = [grp.count(0) / len(grp) for grp in groups]
    return {"payoff": payoff, "pwin": pwin, "pdraw": pdraw, "sims": groups}

def gpt_per_game_payoff(game_data):
    # game_data is a list of (first_adv, draw_response) tuples (0-100 ints).
    first_adv = [x[0] for x in game_data]
    pdraw_raw = [x[1] for x in game_data]
    pwin = [get_pwin(d, f) / 100 for d, f in zip(pdraw_raw, first_adv)]
    pdraw = [x / 100 for x in pdraw_raw]
    payoff = [compute_utility(pd, pw) for pd, pw in zip(pdraw, pwin)]
    return {"payoff": payoff, "pwin": pwin, "pdraw": pdraw, "sims": game_data}

model_payoff = {}

symbolic_models = ["ours", "ours_full", "depth3", "random_terminal",
                   "expert", "expert_mcts", "offense", "defense"]
for model in symbolic_models:
    model_payoff[model] = {}
    for game in ordered_games:
        if game not in model2res[model]:
            continue
        s = model2res[model][game]["game_scores"]
        model_payoff[model][game] = per_participant_judgments(s, model)

for model in ["gpt_nocot", "gpt_cot"]:
    model_payoff[model] = {}
    for game in ordered_games:
        if game not in model2res[model]:
            continue
        model_payoff[model][game] = gpt_per_game_payoff(model2res[model][game])

for model_tag, per_game in external_llm_payoffs.items():
    # Already in {game: {payoff: [...]}} form
    model_payoff[model_tag] = per_game

for model, games in model_payoff.items():
    print(f"{model:>22s}: {len(games)} games")

Payoff distances by game category

In [ ]:
game_type2fmt = {
    "diff-win":            "Second Player \n M-1 to Win",
    "infinity":            "Infinite \n Board",
    "loss":                "M in a Row Loses",
    "no-diag":             "No Diagonal \n Win Allowed",
    "only-diag":           "Only Diagonal \n Win Allowed",
    "player1-2pieces":     "First Player \n Moves 2 Pieces",
    "player1-constraintA": "First Player \n Handicap (A)",
    "player1-constraintB": "First Player \n Handicap (B)",
    "player2-2pieces":     "Second Player \n Moves 2 Pieces",
    "rectangle-board":     "M in a Row\nRectangle",
    "simple-win":          "M in a Row\nSquare",
}

model2name = {
    "ours": "Novice", "ours_full": "Novice (Full)", "depth3": "Depth-3",
    "random_terminal": "Random (Full)",
    "expert": "Depth-5", "expert_mcts": "MCTS",
    "offense": "Offense", "defense": "Defense",
    "gpt_nocot": "GPT (Zero-Shot)", "gpt_cot": "GPT (CoT)",
    "llama3170b-vanilla": "Llama 3.1 70b (Zero-Shot)",
    "llama3170b-cot": "Llama 3.1 70b (CoT)",
    "gpt-o1": "o1",
}

# Human mean payoff per game (single value)
task_df = human_df[human_df.task == "advantage"]
human_mean_payoff = {}
for game in ordered_games:
    row = task_df[task_df.game_id == game].iloc[0]
    vals = []
    for resp in literal_eval(row.all_scores):
        pdraw = resp["draw_response"] / 100
        pwin = get_pwin(resp["draw_response"], resp["firstplayer_response"]) / 100
        vals.append(compute_utility(pdraw, pwin))
    human_mean_payoff[game] = float(np.mean(vals))

def model_mean_payoff(model, game):
    entry = model_payoff[model].get(game)
    if entry is None:
        return None
    if model in {"gpt_nocot", "gpt_cot"} or model in external_llms:
        return float(np.mean(entry["payoff"]))
    # symbolic: use mean win/draw across all raw outcomes for stability
    game_data = model2res[model][game]
    pwin = game_data["win_percent"]
    pdraw = game_data["draw_percent"]
    return float(compute_utility(pdraw, pwin))

viz_models = [m for m in model_payoff.keys() if "mcts" not in m]
rows = []
for game_type, games_in_type in all_game_types.items():
    games_in_type = sorted(games_in_type)
    h_scores = [human_mean_payoff[g] for g in games_in_type]
    for model in viz_models:
        m_scores, h_sub = [], []
        for game, h in zip(games_in_type, h_scores):
            m = model_mean_payoff(model, game)
            if m is None:
                continue
            m_scores.append(m)
            h_sub.append(h)
        if len(m_scores) >= 2:
            corr = pearsonr(m_scores, h_sub)[0]
            rmse = sqrt(mean_squared_error(h_sub, m_scores))
        else:
            corr, rmse = 0.0, 0.0
        for m, h in zip(m_scores, h_sub):
            rows.append({
                "game_type": game_type2fmt[game_type],
                "model": model2name[model],
                "dist": float(abs(m - h)),
                "corr": corr,
                "rmse": rmse,
            })

dist_df = pd.DataFrame(rows)
dist_df.to_csv(f"{OUT_DIR}/payoff_cat_dists.csv", index=False)
print(f"Wrote {OUT_DIR}/payoff_cat_dists.csv: {len(dist_df)} rows")

Save

In [ ]:
# model_payoff: symbolic + GPT advantage + external LLM advantage
out_pth = f"{OUT_DIR}/model_payoff.json"
with open(out_pth, "w") as f:
    json.dump(model_payoff, f)
print(f"Wrote {out_pth}: {len(model_payoff)} models")

# llm_funness: per-game fun ratings for every LLM (GPT, Llama, o1)
llm_funness = {
    "gpt_cot_fun":   model2res["gpt_cot_fun"],
    "gpt_nocot_fun": model2res["gpt_nocot_fun"],
    **external_llm_funness,
}
fun_pth = f"{OUT_DIR}/llm_funness.json"
with open(fun_pth, "w") as f:
    json.dump(llm_funness, f)
print(f"Wrote {fun_pth}: {len(llm_funness)} LLMs")